In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

# Database
def get_data_tickers():
    url = "https://raw.githubusercontent.com/heyhanief/IDX-Data/main/Universe"
    df = pd.read_csv(url, header=None)
    return [f"{t}.JK" for t in df[0].astype(str)]

tickers = get_data_tickers()

# Setup
start_date = "2023-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")
benchmark = "^JKSE"
exchange = "IDX"

benchmark = yf.download(
    benchmark,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)["Close"]

def align_close(stock: pd.Series, ref: pd.Series):
    df = pd.concat([stock, ref], axis=1, join="inner").dropna()
    return df.iloc[:, 0], df.iloc[:, 1]

def period_return(close: pd.Series, bars: int):
    if len(close) < bars:
        return 0
    period_close = close.tail(bars)
    return (period_close.iloc[-1] / period_close.iloc[0]) - 1

def strength(close: pd.Series):
    return (
        0.4 * period_return(close, 63)   # ~3 months (63 trading days)
        + 0.2 * period_return(close, 126) # ~6 months
        + 0.2 * period_return(close, 189) # ~9 months
        + 0.2 * period_return(close, 252) # ~12 months (52 weeks)
    )

def relative_strength(close: pd.Series, ref: pd.Series):
    close, ref = align_close(close, ref)
    rs_stock = strength(close)
    rs_ref = strength(ref)
    return round((1 + rs_stock) / (1 + rs_ref) * 100, 2)

def rs_snapshot(close: pd.Series, ref: pd.Series, months: int):
    shift = months * 21
    if len(close) <= shift:
        return np.nan
    return relative_strength(close.iloc[:-shift], ref.iloc[:-shift])

def rs_percentile(series: pd.Series):
    pct = pd.qcut(
        series.rank(method="first"),
        99,
        labels=False,
        duplicates="drop"
    )
    return pct + 1

# RS Stock
rows = []

for ticker in tickers:
    try:
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            continue

        close = df["Close"]
        info = yf.Ticker(ticker).info

        rs = relative_strength(close, benchmark)
        if rs >= 600:
            continue

        rows.append({
            "Ticker": ticker.replace(".JK", ""),
            "Sector": info.get("sector", "N/A"),
            "Industry": info.get("industry", "N/A"),
            "exchange": exchange,
            "Relative Strength": rs,
            "1 Month": rs_snapshot(close, benchmark, 1),
            "3 Months": rs_snapshot(close, benchmark, 3),
            "6 Months": rs_snapshot(close, benchmark, 6),
        })

    except Exception as e:
        print(f"{ticker}: {e}")

df_stock = pd.DataFrame(rows)

df_stock["Percentile"] = rs_percentile(df_stock["Relative Strength"])
df_stock["1 Month"] = rs_percentile(df_stock["1 Month"])
df_stock["3 Months"] = rs_percentile(df_stock["3 Months"])
df_stock["6 Months"] = rs_percentile(df_stock["6 Months"])

df_stock = df_stock.sort_values("Relative Strength", ascending=False)
df_stock.insert(0, "Rank", range(1, len(df_stock) + 1))

df_stock = df_stock[
    [
        "Rank",
        "Ticker",
        "Sector",
        "Industry",
        "exchange",
        "Relative Strength",
        "Percentile",
        "1 Month",
        "3 Months",
        "6 Months",
    ]
]

# RS Industry
industry_rows = []

for (industry, sector), g in df_stock.groupby(["Industry", "Sector"]):
    if len(g) <= 1:
        continue

    industry_rows.append({
        "Industry": industry,
        "Sector": sector,
        "Relative Strength": round(g["Relative Strength"].mean(), 2),
        "1 Month": round(g["1 Month"].mean(), 2),
        "3 Months": round(g["3 Months"].mean(), 2),
        "6 Months": round(g["6 Months"].mean(), 2),
        "tickers": ",".join(
            g.sort_values("Relative Strength", ascending=False)["Ticker"]
        ),
    })

df_industry = pd.DataFrame(industry_rows)

df_industry["Percentile"] = rs_percentile(df_industry["Relative Strength"])
df_industry["1 Month"] = rs_percentile(df_industry["1 Month"])
df_industry["3 Months"] = rs_percentile(df_industry["3 Months"])
df_industry["6 Months"] = rs_percentile(df_industry["6 Months"])

df_industry = df_industry.sort_values("Relative Strength", ascending=False)
df_industry.insert(0, "Rank", range(1, len(df_industry) + 1))

df_industry = df_industry[
    [
        "Rank",
        "Industry",
        "Sector",
        "Relative Strength",
        "Percentile",
        "1 Month",
        "3 Months",
        "6 Months",
        "tickers",
    ]
]

# Export to Excel (xlsx)
df_stock.to_excel("RS_Stock.xlsx", index=False)
df_industry.to_excel("RS_Industry.xlsx", index=False)
